In [1]:
import sys, time, platform, statistics
import pandas as pd

REPO = "/home/ibm-user/shared/safecircle"
sys.path.insert(0, REPO)
from src.messages import score_message
from src.transactions import score_transaction
from src.risk import decide
from src.explain import redact, explain

print("Running on:", platform.machine())

Running on: s390x


In [2]:
messages     = pd.read_csv(f"{REPO}/scenarios/messages.csv")
transactions = pd.read_csv(f"{REPO}/scenarios/transactions.csv")
messages.columns     = messages.columns.str.strip().str.lower()
transactions.columns = transactions.columns.str.strip().str.lower()

msg_text = messages.set_index("user_id")["text"]
TXN_KEYS = ["user_id", "txn_id", "timestamp", "amount", "merchant",
            "merchant_category", "is_new_payee"]

cases = [(msg_text.get(t["user_id"], ""), t[TXN_KEYS].to_dict())
         for _, t in transactions.iterrows()]
print(len(cases), "scenarios loaded:", [c[1]["user_id"] for c in cases])

3 scenarios loaded: ['rosa', 'walter', 'linda']


In [3]:
N = 1000
steps = {"message model": [], "payment model": [], "decide": [],
         "redact + explain": [], "total": []}

# Warm-up: the first call loads model files from disk, which isn't part of a real decision
text, txn = cases[0]
explain(redact(text), decide(score_message(text), score_transaction(txn)))

for i in range(N):
    text, txn = cases[i % len(cases)]
    t0 = time.perf_counter(); m = score_message(text) if text else {"score": 0.0, "reasons": []}
    t1 = time.perf_counter(); t = score_transaction(txn)
    t2 = time.perf_counter(); d = decide(m, t)
    t3 = time.perf_counter(); explain(redact(text), d)
    t4 = time.perf_counter()
    for name, a, b in [("message model", t0, t1), ("payment model", t1, t2),
                       ("decide", t2, t3), ("redact + explain", t3, t4), ("total", t0, t4)]:
        steps[name].append((b - a) * 1000)

speed = pd.DataFrame({
    name: {"median ms": statistics.median(v),
           "slowest 5% ms": sorted(v)[int(0.95 * N)],
           "share of total": statistics.median(v) / statistics.median(steps["total"])}
    for name, v in steps.items()}).T
speed["share of total"] = (speed["share of total"] * 100).round(0).astype(int).astype(str) + "%"
speed[["median ms", "slowest 5% ms"]] = speed[["median ms", "slowest 5% ms"]].round(3)

print(f"{N} payment decisions on {platform.machine()} (IBM LinuxONE)")
print(f"Decisions per second: {1000 / statistics.median(steps['total']):,.0f}")
speed

1000 payment decisions on s390x (IBM LinuxONE)
Decisions per second: 143


,median ms,slowest 5% ms,share of total
message model,0.826,0.928,12%
payment model,6.177,6.662,88%
decide,0.004,0.005,0%
redact + explain,0.043,0.054,1%
total,6.996,7.551,100%


In [4]:
import os, glob, importlib.metadata
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import tensorflow as tf

print("TensorFlow:", tf.__version__)
try:
    print("ibm-zdnn-plugin:", importlib.metadata.version("ibm-zdnn-plugin"))
except importlib.metadata.PackageNotFoundError:
    print("ibm-zdnn-plugin: NOT INSTALLED in this kernel")
print("zDNN kernel library:", glob.glob("/opt/venv/lib*/python3*/site-packages/tensorflow/core/kernels/_z_ops.so") or "not found")
print("Devices TensorFlow sees:", tf.config.list_physical_devices())

TensorFlow: 2.18.1
ibm-zdnn-plugin: 2.3.0
zDNN kernel library: ['/opt/venv/lib/python3.12/site-packages/tensorflow/core/kernels/_z_ops.so', '/opt/venv/lib64/python3.12/site-packages/tensorflow/core/kernels/_z_ops.so']
Devices TensorFlow sees: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


In [5]:
import joblib, numpy as np, time, statistics

pipe  = joblib.load(f"{REPO}/models/message_model.joblib")
tfidf = pipe.named_steps["tfidf"]
clf   = pipe.named_steps["clf"]

W = clf.coef_.T.astype("float32")
b = clf.intercept_.astype("float32")
tf_model = tf.keras.Sequential([tf.keras.Input(shape=(W.shape[0],)),
                                tf.keras.layers.Dense(1, activation="sigmoid")])
tf_model.layers[0].set_weights([W, b])
print("Vocabulary size:", W.shape[0])

texts = (list(msg_text.values) * 700)[:2000]
X_sparse = tfidf.transform(texts)
X = X_sparse.toarray().astype("float32")

p_sklearn = clf.predict_proba(X_sparse)[:, 1]
p_tf      = tf_model(X, training=False).numpy()[:, 0]
print("Same answers? largest difference:", float(np.abs(p_sklearn - p_tf).max()))

def time_ms(fn, repeats=20):
    fn()
    runs = []
    for _ in range(repeats):
        t0 = time.perf_counter(); fn(); runs.append((time.perf_counter() - t0) * 1000)
    return statistics.median(runs)

print(f"scikit-learn (CPU), 2,000 messages:  {time_ms(lambda: clf.predict_proba(X_sparse)):.2f} ms")
print(f"TensorFlow + zDNN, 2,000 messages:   {time_ms(lambda: tf_model(X, training=False).numpy()):.2f} ms")

Vocabulary size: 12275
Same answers? largest difference: 4.6400291942028105e-08
scikit-learn (CPU), 2,000 messages:  0.16 ms
TensorFlow + zDNN, 2,000 messages:   77.67 ms


In [6]:
%%writefile /home/ibm-user/shared/safecircle/notebooks/phase3/z2_nnpa_check.py
import os, joblib
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import tensorflow as tf
pipe = joblib.load("/home/ibm-user/shared/safecircle/models/message_model.joblib")
clf = pipe.named_steps["clf"]
W, b = clf.coef_.T.astype("float32"), clf.intercept_.astype("float32")
model = tf.keras.Sequential([tf.keras.Input(shape=(W.shape[0],)),
                             tf.keras.layers.Dense(1, activation="sigmoid")])
model.layers[0].set_weights([W, b])
X = pipe.named_steps["tfidf"].transform(["Grandma it's me, please buy gift cards"] * 2000).toarray().astype("float32")
print("scores:", model(X, training=False).numpy()[:3, 0])

Overwriting /home/ibm-user/shared/safecircle/notebooks/phase3/z2_nnpa_check.py


In [7]:
from tensorflow.python.framework import kernels

for op in ["MatMul", "BiasAdd", "Sigmoid"]:
    entries = {(k.device_type, k.label or "-", k.priority)
               for k in kernels.get_registered_kernels_for_op(op).kernel}
    print(f"{op:8} -> {sorted(entries)}")

MatMul   -> [('CPU', '-', 0), ('XLA_CPU_JIT', '-', 0)]
BiasAdd  -> [('CPU', '-', 0), ('XLA_CPU_JIT', '-', 0)]
Sigmoid  -> [('CPU', '-', 0), ('XLA_CPU_JIT', '-', 0)]


In [8]:
import numpy as np
rng = np.random.default_rng(0)
A = rng.standard_normal((2048, 2048)).astype("float32")
B = rng.standard_normal((2048, 2048)).astype("float32")

exact  = A.astype("float64") @ B.astype("float64")      # very precise reference
np_out = A @ B                                          # NumPy on the regular CPU
tf_out = tf.linalg.matmul(A, B).numpy()                 # TensorFlow (+ zDNN plugin)

rel_err = lambda x: float(np.abs(x - exact).max() / np.abs(exact).max())
print(f"NumPy (CPU)        relative error: {rel_err(np_out):.1e}")
print(f"TensorFlow + zDNN  relative error: {rel_err(tf_out):.1e}")

# Same check on OUR message model (needs Cell B run first)
X64 = X.astype("float64")
exact_p = 1 / (1 + np.exp(-(X64 @ clf.coef_.T.astype("float64") + clf.intercept_)))[:, 0]
print(f"\nMessage model, TensorFlow vs exact: {np.abs(p_tf - exact_p).max():.1e}")
print(f"Message model, sklearn vs exact:    {np.abs(p_sklearn - exact_p).max():.1e}")

NumPy (CPU)        relative error: 6.2e-07
TensorFlow + zDNN  relative error: 3.6e-07

Message model, TensorFlow vs exact: 4.7e-08
Message model, sklearn vs exact:    8.6e-09


In [9]:
import time, statistics

def best_ms(fn, repeats=5):
    fn()                                         # warm-up
    runs = []
    for _ in range(repeats):
        t0 = time.perf_counter(); fn(); runs.append((time.perf_counter() - t0) * 1000)
    return min(runs)

np_ms = best_ms(lambda: A @ B)
tf_ms = best_ms(lambda: tf.linalg.matmul(A, B).numpy())
print(f"2048×2048 matrix multiply")
print(f"  NumPy (CPU):        {np_ms:8.1f} ms")
print(f"  TensorFlow + zDNN:  {tf_ms:8.1f} ms   ({np_ms / tf_ms:.1f}× faster)")

2048×2048 matrix multiply
  NumPy (CPU):           314.1 ms
  TensorFlow + zDNN:     107.9 ms   (2.9× faster)


In [10]:
@tf.function
def graph_matmul(a, b):
    return tf.linalg.matmul(a, b)

g_out = graph_matmul(A, B).numpy()
print(f"Graph-mode TensorFlow relative error: {rel_err(g_out):.1e}")
print(f"Graph-mode matmul: {best_ms(lambda: graph_matmul(A, B).numpy()):.1f} ms")

g_pred = tf_model.predict(X, batch_size=2000, verbose=0)[:, 0]   # Keras predict = graph mode
print(f"Message model via predict() vs exact: {np.abs(g_pred - exact_p).max():.1e}")

Graph-mode TensorFlow relative error: 4.9e-03
Graph-mode matmul: 10.1 ms
Message model via predict() vs exact: 4.5e-04


In [11]:
same = ((g_pred >= 0.5) == (p_sklearn >= 0.5)).mean()
print(f"Same scam / not-scam call as scikit-learn: {same:.1%}\n")

texts3 = list(msg_text.values)
sk3 = clf.predict_proba(tfidf.transform(texts3))[:, 1]
nn3 = tf_model.predict(tfidf.transform(texts3).toarray().astype("float32"), verbose=0)[:, 0]
for user, a, b in zip(msg_text.index, sk3, nn3):
    print(f"{user:7} scikit-learn {a:.4f}   NNPA {b:.4f}   difference {abs(a - b):.4f}")

Same scam / not-scam call as scikit-learn: 100.0%

rosa    scikit-learn 0.5882   NNPA 0.5879   difference 0.0003
walter  scikit-learn 0.2940   NNPA 0.2939   difference 0.0001
linda   scikit-learn 0.7076   NNPA 0.7080   difference 0.0005
